# -*- coding: utf-8 -*-
"""
===================================================================================================
PROJECT: REAL-TIME VIDEO IMAGE CLASSIFICATION USING ViT (vit_qat_int8)
         SIMULATING AMD KRIA KV260 HARDWARE ACCELERATOR (ARM PS + FPGA PL)
===================================================================================================
WHAT:
    Ứng dụng phân loại hình ảnh thời gian thực từ USB Webcam 720p @ 30 FPS sử dụng mô hình
    Vision Transformer lượng tử hóa nhận biết (QAT INT8: vit_qat_int8.pth) trên môi trường
    mô phỏng bo mạch nhúng AMD Kria KV260 Starter Kit.

WHY:
    - Bo mạch AMD Kria KV260 là nền tảng Edge AI tiêu chuẩn kết hợp ARM Cortex-A53 (PS)
      và FPGA Programmable Logic (PL) để tăng tốc mạng nơ-ron với độ trễ thấp và tiết kiệm năng lượng.
    - Ứng dụng này giải quyết trọn vẹn luồng dữ liệu thực tế (End-to-End Edge AI Pipeline):
      * Thu thập camera USB 720p không trễ buffer (Threaded Video Stream).
      * Phân chia khối lượng công việc (HW/SW Partitioning):
        - ARM PS: Tiền xử lý khung hình (Center ROI Crop, bicubic 32x32, CIFAR Norm),
                  hậu xử lý MLP Head và render giao diện đồ họa HUD.
        - FPGA PL: Mô phỏng khối tăng tốc phần cứng Attention Core (Systolic Array,
                   ASR Scaler, LUT Softmax) và giao tiếp bộ nhớ AXI DMA / CMA BRAM.
      * Trực quan hóa kết quả phân loại và bảng phân rã độ trễ phần cứng chuẩn công nghiệp.

FUNCTIONALITY:
    1. Tự động kiểm tra và nạp trọng số INT8 từ ./models_cache/vit_qat_int8.pth.
    2. Đọc camera 720p @ 30 FPS đa luồng, tự động kích hoạt Synthetic Fallback nếu không có camera.
    3. Cắt vùng Center ROI 720x720 và nạp vào mô hình ViT 32x32.
    4. Render giao diện Edge AI HUD thời gian thực với khung nhắm mục tiêu, Top-3 Bar Charts,
       chỉ số FPS và bảng phân tích trễ: T_prep, T_dma, T_pl, T_post.
    5. Cung cấp các phím tắt: 'q' (Thoát), 's' (Chụp ảnh), 'd' (Bật/Tắt Debug), 'f' (Khóa 30 FPS).
===================================================================================================
"""


# PHẦN 1: Cấu Hình Môi Trường & Auto-Install Thư Viện Phụ Thuộc
Kiểm tra và tự động cài đặt các thư viện cần thiết (`numpy`, `torch`, `torchvision`, `opencv-python`).


In [1]:
import sys
import subprocess
import importlib.util

# Cấu hình UTF-8 console Windows
if sys.platform.startswith('win'):
    try:
        sys.stdout.reconfigure(encoding='utf-8', errors='replace')
        sys.stderr.reconfigure(encoding='utf-8', errors='replace')
    except Exception:
        pass

required_packages = {
    'numpy': 'numpy',
    'torch': 'torch',
    'torchvision': 'torchvision',
    'cv2': 'opencv-python'
}
missing = [pip_name for mod, pip_name in required_packages.items() if importlib.util.find_spec(mod) is None]

if missing:
    print(f"[Setup] Đang tự động cài đặt các gói còn thiếu: {missing}")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
    print("[Setup] Hoàn tất cài đặt dependencies!")
else:
    print("[Setup] Tất cả các thư viện phụ thuộc đã sẵn sàng.")

import os
import time
import math
import threading
from datetime import datetime
from typing import Tuple, List, Dict, Optional, Union

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.ao.quantization as quantization

# Cố định seed ngẫu nhiên
torch.manual_seed(42)
np.random.seed(42)


[Setup] Tất cả các thư viện phụ thuộc đã sẵn sàng.


# PHẦN 2: Siêu Tham Số Kiến Trúc ViT & Cấu Hình Backend Lượng Tử Hóa
Thiết lập các thông số khớp chính xác với mô hình `vit_qat_int8.pth` được huấn luyện trên CIFAR-10:
- Độ phân giải ảnh: $32 \times 32 \times 3$
- Kích thước patch: $4 \times 4 \implies 64$ patches (+ 1 CLS token = 65 tokens)
- Số heads: 8 ($d_k = 32$), Chiều nhúng: 256, Độ sâu: 10 tầng Encoder


In [2]:
IMG_SIZE = 32         # 32x32 pixels
NUM_CHANNELS = 3      # 3 kênh RGB
NUM_CLASSES = 10      # 10 lớp CIFAR-10
PATCH_SIZE = 4        # 4x4 pixels
NUM_PATCHES = (IMG_SIZE // PATCH_SIZE) ** 2  # 64 patches
NUM_HEADS = 8         # 8 Attention Heads
EMBED_DIM = 256       # 256 embedding dimension
HEAD_DIM = EMBED_DIM // NUM_HEADS  # d_k = 32
MLP_DIM = 512         # 512 hidden dimension
DROP_RATE = 0.1       # Dropout 10%
DEPTH = 10            # 10 Transformer Encoder Blocks

# 10 nhãn phân loại chuẩn CIFAR-10
CIFAR10_CLASSES = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
]

# Giá trị Mean và Std của bộ dữ liệu CIFAR-10
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)

def setup_quantization_engine() -> str:
    """Tự động lựa chọn quantization engine phù hợp với CPU (onednn / fbgemm / qnnpack)."""
    supported_engines = torch.backends.quantized.supported_engines
    chosen_engine = None

    if 'onednn' in supported_engines:
        chosen_engine = 'onednn'
    elif 'fbgemm' in supported_engines:
        chosen_engine = 'fbgemm'
    elif 'qnnpack' in supported_engines:
        chosen_engine = 'qnnpack'
    else:
        chosen_engine = list(supported_engines)[0] if len(supported_engines) > 0 else 'none'

    torch.backends.quantized.engine = chosen_engine
    print(f"[KV260 Engine] Quantization Backend: '{chosen_engine}' (Hỗ trợ: {supported_engines})")
    return chosen_engine


# PHẦN 3: Định Nghĩa Kiến Trúc Vision Transformer Thân Thiện Phần Cứng
Tái cấu trúc mô hình ViT với các khối `ScratchMultiheadAttention` và `HWFriendlyTransformerEncoder`
nhằm mô phỏng chính xác luồng dữ liệu trên vi mạch FPGA PL Kria KV260.


In [3]:
class PatchEmbedding(nn.Module):
    """Chiếu cắt ảnh thành 64 patch và ánh xạ tuyến tính lên không gian 256 chiều."""
    def __init__(self, num_channels: int = NUM_CHANNELS, embed_dim: int = EMBED_DIM, patch_size: int = PATCH_SIZE):
        super().__init__()
        self.patch_embed = nn.Conv2d(num_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # [B, 3, 32, 32] -> [B, 256, 8, 8] -> [B, 64, 256]
        return self.patch_embed(x).flatten(2).transpose(1, 2)


class MLP(nn.Module):
    """Khối Feed-Forward Network với hàm kích hoạt GELU."""
    def __init__(self, embed_dim: int = EMBED_DIM, mlp_dim: int = MLP_DIM, drop_rate: float = DROP_RATE):
        super().__init__()
        self.linear1 = nn.Linear(embed_dim, mlp_dim)
        self.gelu = nn.GELU()
        self.linear2 = nn.Linear(mlp_dim, embed_dim)
        self.dropout = nn.Dropout(drop_rate)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.dropout(self.linear2(self.dropout(self.gelu(self.linear1(x)))))


class ScratchMultiheadAttention(nn.Module):
    """
    WHAT: Khối Attention tự triển khai mô phỏng 4 công đoạn tính toán phần cứng FPGA PL:
    - Stage 1: Systolic Array 1 nhân Q * K^T (INT32 Partial Sums).
    - Stage 2: Scaler Unit nhân 1/sqrt(d_k) (dịch bit phải số học ASR).
    - Stage 3: Hardware Softmax (tra bảng LUT ép về xác suất INT8).
    - Stage 4: Systolic Array 2 nhân Score * V (INT8 x INT8).
    """
    def __init__(self, embed_dim: int = EMBED_DIM, num_heads: int = NUM_HEADS, dropout: float = DROP_RATE):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads  # d_k = 32

        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)

        self.dropout = nn.Dropout(dropout)
        self.scale = 1.0 / (self.head_dim ** 0.5)

        self.ff_matmul_qv = nn.quantized.FloatFunctional()
        self.attn_probs_quant = quantization.QuantStub()

    def forward(self, q: torch.Tensor, k: torch.Tensor, v: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        B, N, C = q.shape

        q_proj = self.q_proj(q)
        k_proj = self.k_proj(k)
        v_proj = self.v_proj(v)

        q_heads = q_proj.reshape(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        k_heads = k_proj.reshape(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        v_heads = v_proj.reshape(B, N, self.num_heads, self.head_dim).transpose(1, 2)

        q_heads_f = q_heads.dequantize() if q_heads.is_quantized else q_heads
        k_heads_f = k_heads.dequantize() if k_heads.is_quantized else k_heads

        # STAGE 1: Q * K^T
        attn_scores = torch.matmul(q_heads_f, k_heads_f.transpose(-2, -1))

        # STAGE 2: Scaler Unit
        attn_scores = attn_scores * self.scale

        # STAGE 3: Hardware Softmax LUT
        attn_probs = F.softmax(attn_scores, dim=-1)
        attn_probs = self.dropout(attn_probs)
        attn_probs = self.attn_probs_quant(attn_probs)

        # STAGE 4: Score * V
        context = self.ff_matmul_qv.matmul(attn_probs, v_heads)
        context = context.transpose(1, 2).reshape(B, N, C)
        output = self.out_proj(context)

        return output, attn_probs


class HWFriendlyTransformerEncoder(nn.Module):
    """Transformer Encoder với phép cộng Residual lượng tử hóa an toàn bằng FloatFunctional."""
    def __init__(self, embed_dim: int = EMBED_DIM, num_heads: int = NUM_HEADS,
                 mlp_dim: int = MLP_DIM, drop_rate: float = DROP_RATE):
        super().__init__()
        self.layer_norm_1 = nn.LayerNorm(embed_dim)
        self.multi_head_atten = ScratchMultiheadAttention(embed_dim, num_heads, dropout=drop_rate)
        self.layer_norm_2 = nn.LayerNorm(embed_dim)
        self.multi_layer_perceptron = MLP(embed_dim, mlp_dim, drop_rate)

        self.ff_residual_1 = nn.quantized.FloatFunctional()
        self.ff_residual_2 = nn.quantized.FloatFunctional()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual_1 = x
        attention_output, _ = self.multi_head_atten(self.layer_norm_1(x), self.layer_norm_1(x), self.layer_norm_1(x))
        x = self.ff_residual_1.add(attention_output, residual_1)

        residual_2 = x
        mlp_output = self.multi_layer_perceptron(self.layer_norm_2(x))
        x = self.ff_residual_2.add(mlp_output, residual_2)
        return x


class QuantizableVisionTransformer(nn.Module):
    """Mô hình Vision Transformer lượng tử hóa QAT INT8 hoàn chỉnh."""
    def __init__(self, img_size: int = IMG_SIZE, patch_size: int = PATCH_SIZE,
                 num_channels: int = NUM_CHANNELS, num_classes: int = NUM_CLASSES,
                 embed_dim: int = EMBED_DIM, depth: int = DEPTH, num_heads: int = NUM_HEADS,
                 mlp_dim: int = MLP_DIM, drop_rate: float = DROP_RATE):
        super().__init__()
        num_patches = (img_size // patch_size) ** 2
        self.patch_embedding = PatchEmbedding(num_channels, embed_dim, patch_size)
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.randn(1, 1 + num_patches, embed_dim))

        self.transformer_layers = nn.Sequential(*[
            HWFriendlyTransformerEncoder(embed_dim, num_heads, mlp_dim, drop_rate)
            for _ in range(depth)
        ])

        self.mlp_head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, num_classes)
        )

        self.quant = quantization.QuantStub()
        self.dequant = quantization.DeQuantStub()
        self.quant_cls = quantization.QuantStub()
        self.quant_pos = quantization.QuantStub()
        self.f_cat = nn.quantized.FloatFunctional()
        self.f_add = nn.quantized.FloatFunctional()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.quant(x)
        x = self.patch_embedding(x)
        B = x.size(0)

        cls_tokens = self.cls_token.expand(B, -1, -1)
        cls_tokens_quant = self.quant_cls(cls_tokens)
        x = self.f_cat.cat((cls_tokens_quant, x), dim=1)

        pos_embed_quant = self.quant_pos(self.pos_embed)
        x = self.f_add.add(x, pos_embed_quant)

        x = self.transformer_layers(x)
        logits = self.mlp_head(x[:, 0])
        return self.dequant(logits)


# PHẦN 4: Bộ Nạp Mô Hình QAT INT8 & Giả Lập Phần Cứng Kria KV260
Tải file trọng số `vit_qat_int8.pth` và khởi tạo đối tượng `KriaKV260Simulator` mô phỏng:
- Cấp phát bộ nhớ CMA vật lý liên tục (`pynq.allocate`).
- Đo lường độ trễ truyền dữ liệu AXI DMA (1.6 GB/s).
- Tính toán chu kỳ xung nhịp phần cứng mảng Systolic MAC trên FPGA PL.


In [ ]:
def load_vit_qat_model(weights_path: str = "models_cache/vit_qat_int8.pth") -> nn.Module:
    """Nạp trọng số INT8 vào mô hình QuantizableVisionTransformer."""
    engine = setup_quantization_engine()

    possible_paths = [
        weights_path,
        os.path.join(os.path.dirname(__file__), weights_path) if "__file__" in locals() else weights_path,
        os.path.join("model_notebook", weights_path),
        os.path.join("..", weights_path),
        os.path.join("e:/CapstoneProjectDocs/model_notebook", weights_path),
        os.path.join("e:/CapstoneProjectDocs/model_notebook/models_cache", os.path.basename(weights_path))
    ]

    actual_path = None
    for p in possible_paths:
        if os.path.exists(p):
            actual_path = p
            break

    if actual_path is None:
        raise FileNotFoundError(f"[ERROR] Không tìm thấy file trọng số {weights_path}")

    qat_model = QuantizableVisionTransformer().to('cpu')
    qat_model.train()
    qat_model.qconfig = quantization.get_default_qat_qconfig(engine)
    quantization.prepare_qat(qat_model, inplace=True)
    qat_model_int8 = quantization.convert(qat_model, inplace=False)

    state_dict = torch.load(actual_path, map_location='cpu')
    qat_model_int8.load_state_dict(state_dict)
    qat_model_int8.eval()
    print(f"[KV260 Loader] ✅ Đã nạp thành công '{actual_path}' ({len(state_dict)} tensors).")
    return qat_model_int8


class KriaKV260Simulator:
    """Bộ mô phỏng vi mạch Kria KV260: AXI DMA, CMA BRAM và phân tách độ trễ."""
    def __init__(self, model_weights_path: str = "models_cache/vit_qat_int8.pth"):
        self.pl_clock_freq_mhz = 200.0
        self.clock_period_ns = 1000.0 / self.pl_clock_freq_mhz
        self.dma_bandwidth_gbps = 1.6
        self.dma_overhead_ms = 0.035
        # self.systolic_dim = 16
        self.systolic_dim = 4
        self.macs_per_cycle = self.systolic_dim * self.systolic_dim  # 256 MACs/cycle

        self.cma_buffer_size = 3 * 65 * 256  # 49,920 bytes (~48.75 KB)
        self.cma_memory_pool = np.zeros(self.cma_buffer_size, dtype=np.int8)

        self.model = load_vit_qat_model(model_weights_path)
        self.frame_counter = 0

    def calculate_pl_hardware_cycles(self, num_tokens: int = 65, head_dim: int = 32,
                                     num_heads: int = 8, depth: int = 10) -> Tuple[int, float]:
        macs_per_layer = 2 * (num_tokens * num_tokens * head_dim * num_heads)
        systolic_cycles_per_layer = math.ceil(macs_per_layer / self.macs_per_cycle)
        softmax_lut_cycles = math.ceil((num_tokens * num_tokens * num_heads) / 16)
        total_cycles_per_layer = systolic_cycles_per_layer + softmax_lut_cycles + 32
        total_pl_cycles = total_cycles_per_layer * depth
        pl_time_ms = (total_pl_cycles * self.clock_period_ns) / 1e6
        return total_pl_cycles, pl_time_ms

    def calculate_dma_transfer_time(self, data_bytes: int) -> float:
        return ((data_bytes / (self.dma_bandwidth_gbps * 1e9)) * 1000.0) + self.dma_overhead_ms

    def predict_tensor(self, input_tensor: torch.Tensor, t_prep_ms: float = 1.5) -> Tuple[int, float, List[Tuple[str, float]], Dict[str, float]]:
        self.frame_counter += 1
        t_start = time.perf_counter()

        t_infer_start = time.perf_counter()
        with torch.no_grad():
            logits = self.model(input_tensor)
            probabilities = F.softmax(logits, dim=1).squeeze(0)
        measured_infer_ms = (time.perf_counter() - t_infer_start) * 1000.0

        top_prob, top_idx = torch.max(probabilities, dim=0)
        top1_class_id = int(top_idx.item())
        top1_confidence = float(top_prob.item())

        top3_probs, top3_indices = torch.topk(probabilities, k=3)
        top3_list = [(CIFAR10_CLASSES[idx.item()], float(prob.item())) for prob, idx in zip(top3_probs, top3_indices)]

        pl_cycles, theoretical_pl_ms = self.calculate_pl_hardware_cycles(num_tokens=65, depth=10)
        dma_tx_ms = self.calculate_dma_transfer_time(self.cma_buffer_size)
        dma_rx_ms = self.calculate_dma_transfer_time(65 * 256)
        total_dma_ms = dma_tx_ms + dma_rx_ms

        ps_mlp_head_ms = max(0.2, measured_infer_ms * 0.25)
        simulated_pl_ms = max(theoretical_pl_ms, measured_infer_ms * 0.65)
        total_e2e_ms = (time.perf_counter() - t_start) * 1000.0 + t_prep_ms

        latency_breakdown = {
            "t_prep_ms": t_prep_ms,
            "t_dma_total_ms": total_dma_ms,
            "t_pl_attention_ms": simulated_pl_ms,
            "t_ps_mlp_head_ms": ps_mlp_head_ms,
            "t_measured_infer_ms": measured_infer_ms,
            "t_e2e_ms": total_e2e_ms,
            "pl_hardware_cycles": pl_cycles
        }
        return top1_class_id, top1_confidence, top3_list, latency_breakdown


# PHẦN 5: Pipeline Thu Thập Video 720p @ 30 FPS & Tiền Xử Lý ARM PS
- `ThreadedCameraStream`: Thu thập khung hình ngầm chống nghẽn buffer. Hỗ trợ Synthetic Fallback.
- `PSPreprocessor`: Cắt Center ROI 720x720, co ảnh 32x32 và chuẩn hóa CIFAR-10.


In [5]:
class SyntheticFrameGenerator:
    """Sinh luồng video giả lập 720p @ 30 FPS khi không có camera vật lý."""
    def __init__(self, width: int = 1280, height: int = 720):
        self.width = width
        self.height = height
        self.frame_idx = 0
        self.classes = ['frog', 'airplane', 'automobile', 'bird', 'ship', 'horse', 'dog', 'cat']
        self.cur_idx = 0
        self.last_switch = time.time()

    def generate_frame(self) -> np.ndarray:
        self.frame_idx += 1
        now = time.time()
        if now - self.last_switch > 4.0:
            self.cur_idx = (self.cur_idx + 1) % len(self.classes)
            self.last_switch = now

        current_class = self.classes[self.cur_idx]
        frame = np.full((self.height, self.width, 3), 25, dtype=np.uint8)

        # Lưới tọa độ đồ họa
        for x in range(0, self.width, 80): cv2.line(frame, (x, 0), (x, self.height), (35, 35, 35), 1)
        for y in range(0, self.height, 80): cv2.line(frame, (0, y), (self.width, y), (35, 35, 35), 1)

        cx = self.width // 2 + int(np.sin(self.frame_idx * 0.05) * 40)
        cy = self.height // 2 + int(np.cos(self.frame_idx * 0.05) * 30)

        if current_class == 'frog':
            cv2.circle(frame, (cx, cy), 110, (40, 180, 50), -1)
            cv2.circle(frame, (cx - 45, cy - 80), 30, (50, 200, 60), -1)
            cv2.circle(frame, (cx + 45, cy - 80), 30, (50, 200, 60), -1)
        elif current_class in ['airplane', 'bird']:
            pts = np.array([[cx, cy - 120], [cx + 140, cy], [cx, cy + 100], [cx - 140, cy]], np.int32)
            cv2.fillPoly(frame, [pts], (220, 180, 70))
        elif current_class in ['automobile', 'truck']:
            cv2.rectangle(frame, (cx - 140, cy - 50), (cx + 140, cy + 70), (40, 50, 220), -1)
            cv2.rectangle(frame, (cx - 90, cy - 110), (cx + 90, cy - 50), (60, 70, 240), -1)
            cv2.circle(frame, (cx - 80, cy + 70), 28, (30, 30, 30), -1)
            cv2.circle(frame, (cx + 80, cy + 70), 28, (30, 30, 30), -1)
        else:
            cv2.circle(frame, (cx, cy), 110, (180, 120, 70), -1)

        cv2.putText(frame, "[SYNTHETIC CAMERA FALLBACK - 720p @ 30FPS]", (30, 45),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 255, 255), 2, cv2.LINE_AA)
        cv2.putText(frame, f"Simulated Object: {current_class.upper()} (Auto-switch 4s)", (30, 80),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (180, 180, 180), 1, cv2.LINE_AA)
        return frame


class ThreadedCameraStream:
    """Luồng đọc camera ngầm hiệu năng cao."""
    def __init__(self, camera_index: int = 0, width: int = 1280, height: int = 720, fps: int = 30):
        self.camera_index = camera_index
        self.width = width
        self.height = height
        self.fps = fps

        self.cap: Optional[cv2.VideoCapture] = None
        self.is_synthetic = False
        self.synthetic_gen: Optional[SyntheticFrameGenerator] = None

        self.latest_frame: Optional[np.ndarray] = None
        self.is_running = False
        self.lock = threading.Lock()
        self.thread: Optional[threading.Thread] = None

        self._init_stream()

    def _init_stream(self):
        print(f"[Camera] Tìm kiếm USB Webcam tại index {self.camera_index}...")
        cap = cv2.VideoCapture(self.camera_index, cv2.CAP_DSHOW) if sys.platform.startswith('win') else cv2.VideoCapture(self.camera_index)
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, self.width)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, self.height)
        cap.set(cv2.CAP_PROP_FPS, self.fps)

        success, test_frame = cap.read() if cap.isOpened() else (False, None)
        if success and test_frame is not None:
            self.cap = cap
            self.is_synthetic = False
            self.latest_frame = test_frame
            print(f"[Camera] ✅ Kết nối thành công USB Webcam ({int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))}x{int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))} @ {cap.get(cv2.CAP_PROP_FPS):.1f} FPS)")
        else:
            if cap.isOpened(): cap.release()
            self.cap = None
            self.is_synthetic = True
            self.synthetic_gen = SyntheticFrameGenerator(self.width, self.height)
            self.latest_frame = self.synthetic_gen.generate_frame()
            print(f"[Camera] 🚀 Chuyển sang Synthetic Fallback Mode (720p @ 30 FPS).")

    def start(self) -> "ThreadedCameraStream":
        if self.is_running: return self
        self.is_running = True
        self.thread = threading.Thread(target=self._worker, daemon=True, name="CamWorker")
        self.thread.start()
        return self

    def _worker(self):
        fps_interval = 1.0 / self.fps
        last_t = time.time()
        while self.is_running:
            if not self.is_synthetic and self.cap is not None:
                ret, frame = self.cap.read()
                if ret and frame is not None:
                    with self.lock: self.latest_frame = frame
                else:
                    self.is_synthetic = True
                    self.synthetic_gen = SyntheticFrameGenerator(self.width, self.height)
            else:
                now = time.time()
                elapsed = now - last_t
                if elapsed < fps_interval: time.sleep(fps_interval - elapsed)
                last_t = time.time()
                frame = self.synthetic_gen.generate_frame()
                with self.lock: self.latest_frame = frame

    def read(self) -> Tuple[bool, Optional[np.ndarray]]:
        with self.lock:
            return (True, self.latest_frame.copy()) if self.latest_frame is not None else (False, None)

    def stop(self):
        self.is_running = False
        if self.thread: self.thread.join(timeout=1.0)
        if self.cap and self.cap.isOpened(): self.cap.release()


class PSPreprocessor:
    """Tiền xử lý khung hình trên lõi ARM PS: Crop Center ROI và chuẩn hóa CIFAR-10."""
    def __init__(self, target_size: int = IMG_SIZE):
        self.target_size = target_size
        self.mean = np.array(CIFAR_MEAN, dtype=np.float32).reshape(1, 1, 3)
        self.std = np.array(CIFAR_STD, dtype=np.float32).reshape(1, 1, 3)

    def crop_center_roi(self, frame: np.ndarray) -> Tuple[np.ndarray, Tuple[int, int, int, int]]:
        h, w = frame.shape[:2]
        roi_dim = min(h, w)
        x1 = (w - roi_dim) // 2
        y1 = (h - roi_dim) // 2
        return frame[y1:y1 + roi_dim, x1:x1 + roi_dim], (x1, y1, x1 + roi_dim, y1 + roi_dim)

    def preprocess_to_tensor(self, roi_img: np.ndarray) -> Tuple[torch.Tensor, float]:
        t0 = time.perf_counter()
        resized = cv2.resize(roi_img, (self.target_size, self.target_size), interpolation=cv2.INTER_AREA)
        rgb = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)
        norm = (rgb.astype(np.float32) / 255.0 - self.mean) / self.std
        tensor = torch.from_numpy(np.transpose(norm, (2, 0, 1))).unsqueeze(0)
        return tensor, (time.perf_counter() - t0) * 1000.0


# PHẦN 6: Giao Diện Edge AI HUD & Bảng Đo Đạc Phần Cứng Kria KV260
Trực quan hóa vùng ngắm mục tiêu, bảng phân loại Top-3 và bảng phân tích độ trễ.


In [6]:
class KV260HUDVisualizer:
    """Giao diện HUD Overlay thời gian thực phong cách công nghệ Edge AI."""
    def __init__(self, output_dir: str = "captures"):
        self.output_dir = output_dir
        os.makedirs(self.output_dir, exist_ok=True)
        self.debug_mode = True
        self.fps_lock_30 = False
        self.is_paused = False
        self.fps_history: List[float] = []

    def get_smooth_fps(self, fps: float) -> float:
        self.fps_history.append(fps)
        if len(self.fps_history) > 20: self.fps_history.pop(0)
        return float(np.mean(self.fps_history))

    def render(self, frame: np.ndarray, roi_coords: Tuple[int, int, int, int],
               top1_name: str, top1_conf: float, top3_list: List[Tuple[str, float]],
               latency_dict: Dict[str, float], current_fps: float = 30.0) -> np.ndarray:
        h, w = frame.shape[:2]
        canvas = frame.copy()

        # 1. Thanh tiêu đề trên cùng
        overlay = canvas.copy()
        cv2.rectangle(overlay, (0, 0), (w, 42), (18, 18, 22), -1)
        cv2.addWeighted(overlay, 0.85, canvas, 0.15, 0, canvas)
        cv2.putText(canvas, "AMD KRIA KV260 VISION AI STARTER KIT", (20, 27), cv2.FONT_HERSHEY_DUPLEX, 0.65, (0, 230, 255), 1, cv2.LINE_AA)
        cv2.putText(canvas, "[ARM Cortex-A53 PS | FPGA PL INT8 QAT | AXI DMA: 1.6 GB/s]", (490, 27), cv2.FONT_HERSHEY_SIMPLEX, 0.48, (50, 220, 90), 1, cv2.LINE_AA)
        cv2.putText(canvas, datetime.now().strftime("%H:%M:%S"), (w - 95, 27), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (160, 160, 175), 1, cv2.LINE_AA)

        # 2. Khung ngắm Center ROI (Tech Brackets)
        x1, y1, x2, y2 = roi_coords
        c_len = 35
        c_color = (0, 230, 255)
        for (px, py, dx, dy) in [(x1, y1, 1, 1), (x2, y1, -1, 1), (x1, y2, 1, -1), (x2, y2, -1, -1)]:
            cv2.line(canvas, (px, py), (px + dx * c_len, py), c_color, 3)
            cv2.line(canvas, (px, py), (px, py + dy * c_len), c_color, 3)

        cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
        cv2.line(canvas, (cx - 12, cy), (cx + 12, cy), (0, 255, 255), 1)
        cv2.line(canvas, (cx, cy - 12), (cx, cy + 12), (0, 255, 255), 1)
        cv2.putText(canvas, "TARGET REGION (32x32 ViT ROI)", (x1 + 10, y1 + 25), cv2.FONT_HERSHEY_SIMPLEX, 0.5, c_color, 1, cv2.LINE_AA)

        # 3. Bảng Prediction Dashboard
        px, py, pw, ph = 25, 55, 340, 240
        overlay_p = canvas.copy()
        cv2.rectangle(overlay_p, (px, py), (px + pw, py + ph), (18, 18, 22), -1)
        cv2.addWeighted(overlay_p, 0.72, canvas, 0.28, 0, canvas)
        cv2.rectangle(canvas, (px, py), (px + pw, py + ph), (70, 70, 85), 1)

        cv2.putText(canvas, "PREDICTION DASHBOARD", (px + 15, py + 26), cv2.FONT_HERSHEY_DUPLEX, 0.55, (230, 215, 0), 1, cv2.LINE_AA)
        cv2.line(canvas, (px + 15, py + 34), (px + pw - 15, py + 34), (70, 70, 85), 1)

        top_col = (50, 220, 90) if top1_conf >= 0.7 else ((40, 215, 255) if top1_conf >= 0.4 else (40, 50, 235))
        cv2.putText(canvas, top1_name.upper(), (px + 15, py + 92), cv2.FONT_HERSHEY_DUPLEX, 1.0, top_col, 2, cv2.LINE_AA)
        cv2.putText(canvas, f"{top1_conf * 100:.1f}%", (px + pw - 90, py + 92), cv2.FONT_HERSHEY_DUPLEX, 0.85, top_col, 2, cv2.LINE_AA)

        cv2.putText(canvas, "TOP-3 CONFIDENCE BREAKDOWN:", (px + 15, py + 130), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (160, 160, 175), 1, cv2.LINE_AA)
        bar_colors = [(50, 220, 90), (230, 215, 0), (40, 215, 255)]
        for i, (name, prob) in enumerate(top3_list):
            by = py + 150 + (i * 26)
            cv2.putText(canvas, f"{i+1}. {name:<10}", (px + 15, by + 11), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (245, 245, 245), 1, cv2.LINE_AA)
            cv2.rectangle(canvas, (px + 115, by), (px + 275, by + 14), (45, 45, 55), -1)
            cv2.rectangle(canvas, (px + 115, by), (px + 115 + int(160 * prob), by + 14), bar_colors[i % 3], -1)
            cv2.putText(canvas, f"{prob * 100:5.1f}%", (px + 285, by + 11), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (245, 245, 245), 1, cv2.LINE_AA)

        # 4. Bảng Hardware Telemetry
        if self.debug_mode:
            tx, ty, tw, th = w - 375, 55, 350, 360
            overlay_t = canvas.copy()
            cv2.rectangle(overlay_t, (tx, ty), (tx + tw, ty + th), (18, 18, 22), -1)
            cv2.addWeighted(overlay_t, 0.72, canvas, 0.28, 0, canvas)
            cv2.rectangle(canvas, (tx, ty), (tx + tw, ty + th), (70, 70, 85), 1)

            cv2.putText(canvas, "KRIA KV260 HARDWARE TELEMETRY", (tx + 15, ty + 26), cv2.FONT_HERSHEY_DUPLEX, 0.52, (0, 230, 255), 1, cv2.LINE_AA)
            cv2.line(canvas, (tx + 15, ty + 34), (tx + tw - 15, ty + 34), (70, 70, 85), 1)

            s_fps = self.get_smooth_fps(current_fps)
            f_col = (50, 220, 90) if s_fps >= 25 else ((40, 215, 255) if s_fps >= 15 else (40, 50, 235))
            cv2.putText(canvas, f"{s_fps:4.1f} FPS", (tx + 15, ty + 90), cv2.FONT_HERSHEY_DUPLEX, 0.95, f_col, 2, cv2.LINE_AA)
            cv2.putText(canvas, f"[LOCKED 30]" if self.fps_lock_30 else "[UNLOCKED]", (tx + 180, ty + 86), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (180, 180, 180), 1, cv2.LINE_AA)

            rows = [
                ("1. ARM PS Video Preprocess", f"{latency_dict.get('t_prep_ms', 1.5):6.2f} ms", (200, 200, 200)),
                ("2. AXI4-Stream DMA (2-way)", f"{latency_dict.get('t_dma_total_ms', 0.11):6.2f} ms", (0, 220, 255)),
                ("3. FPGA PL Attention Core", f"{latency_dict.get('t_pl_attention_ms', 25.0):6.2f} ms", (50, 220, 90)),
                ("4. ARM PS MLP Head Softmax", f"{latency_dict.get('t_ps_mlp_head_ms', 8.0):6.2f} ms", (200, 200, 200))
            ]
            for idx, (lbl, val, col) in enumerate(rows):
                ry = ty + 150 + (idx * 24)
                cv2.putText(canvas, lbl, (tx + 15, ry), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (245, 245, 245), 1, cv2.LINE_AA)
                cv2.putText(canvas, val, (tx + tw - 95, ry), cv2.FONT_HERSHEY_SIMPLEX, 0.42, col, 1, cv2.LINE_AA)

            tot_lat = latency_dict.get('t_e2e_ms', 35.0)
            cv2.line(canvas, (tx + 15, ty + 258), (tx + tw - 15, ty + 258), (70, 70, 85), 1)
            cv2.putText(canvas, "TOTAL LATENCY (E2E):", (tx + 15, ty + 280), cv2.FONT_HERSHEY_DUPLEX, 0.5, (0, 230, 255), 1, cv2.LINE_AA)
            cv2.putText(canvas, f"{tot_lat:6.2f} ms", (tx + tw - 110, ty + 280), cv2.FONT_HERSHEY_DUPLEX, 0.6, (50, 220, 90), 1, cv2.LINE_AA)
            cv2.putText(canvas, f"PL Compute Cycles: {latency_dict.get('pl_hardware_cycles', 105950):,} cycles", (tx + 15, ty + 325), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (160, 160, 175), 1, cv2.LINE_AA)

        # 5. Thanh phím tắt điều khiển ở đáy
        overlay_b = canvas.copy()
        cv2.rectangle(overlay_b, (0, h - 32), (w, h), (18, 18, 22), -1)
        cv2.addWeighted(overlay_b, 0.85, canvas, 0.15, 0, canvas)
        shortcuts = [("[Q]", "Quit"), ("[S]", "Snapshot"), ("[D]", "Toggle Debug"), ("[F]", "FPS Lock"), ("[P]", "Pause")]
        for i, (k, desc) in enumerate(shortcuts):
            cv2.putText(canvas, k, (25 + i * 230, h - 11), cv2.FONT_HERSHEY_DUPLEX, 0.45, (0, 230, 255), 1, cv2.LINE_AA)
            cv2.putText(canvas, desc, (60 + i * 230, h - 11), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (245, 245, 245), 1, cv2.LINE_AA)

        return canvas

    def save_snapshot(self, frame: np.ndarray, top1_name: str, confidence: float) -> str:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"kv260_snapshot_{timestamp}_{top1_name}_{int(confidence * 100)}pct.png"
        path = os.path.join(self.output_dir, filename)
        cv2.imwrite(path, frame)
        print(f"[Snapshot] 📸 Đã lưu: '{path}'")
        return path


# PHẦN 7: Hàm Điều Khiển Thực Thi Chính (Main Pipeline Execution)
Khởi tạo toàn bộ các thành phần, chạy vòng lặp xử lý video thời gian thực và quản lý phím bấm tương tác.


In [7]:
def run_live_video_classification(camera_index: int = 0, max_frames: Optional[int] = None, display_gui: bool = True):
    """
    WHAT: Hàm chạy toàn bộ ứng dụng phân loại video thời gian thực.
    ARGUMENTS:
        - camera_index: Chỉ số camera USB (mặc định 0).
        - max_frames: Giới hạn số khung hình chạy thử (None nếu chạy vô hạn đến khi bấm 'q').
        - display_gui: True để mở cửa sổ cv2.imshow, False nếu chạy chế độ headless.
    """
    print("\n" + "=" * 80)
    print("  KHỞI ĐỘNG ỨNG DỤNG REAL-TIME VIDEO CLASSIFICATION ViT (AMD KRIA KV260)  ")
    print("=" * 80)

    # 1. Khởi tạo Kria KV260 Simulator & nạp mô hình INT8
    simulator = KriaKV260Simulator()

    # 2. Khởi tạo Camera Stream 720p @ 30 FPS
    stream = ThreadedCameraStream(camera_index=camera_index, width=1280, height=720, fps=30)
    stream.start()

    # 3. Khởi tạo Preprocessor và Visualizer
    preprocessor = PSPreprocessor(target_size=IMG_SIZE)
    visualizer = KV260HUDVisualizer(output_dir="captures")

    time.sleep(0.5)  # Đợi luồng camera ổn định
    window_name = "AMD Kria KV260 - Real-Time ViT Classification (vit_qat_int8)"

    frame_idx = 0
    t_fps_start = time.time()
    fps_measured = 30.0

    print("\n[Application] 🚀 Ứng dụng đã sẵn sàng! Đang phát luồng video...")
    print("              Bấm phím 'q' trên cửa sổ video để thoát.\n")

    try:
        while True:
            t_loop_start = time.perf_counter()

            # Đọc khung hình mới nhất
            ret, frame = stream.read()
            if not ret or frame is None:
                time.sleep(0.01)
                continue

            frame_idx += 1

            if not visualizer.is_paused:
                # 1. ARM PS: Cắt Center ROI và tiền xử lý
                roi, roi_coords = preprocessor.crop_center_roi(frame)
                input_tensor, t_prep_ms = preprocessor.preprocess_to_tensor(roi)

                # 2. FPGA PL & ARM PS: Suy luận mô hình INT8
                top1_id, top1_conf, top3_list, latency_dict = simulator.predict_tensor(input_tensor, t_prep_ms=t_prep_ms)
                top1_name = CIFAR10_CLASSES[top1_id]

                # 3. Render giao diện Edge AI HUD
                rendered_frame = visualizer.render(
                    frame=frame,
                    roi_coords=roi_coords,
                    top1_name=top1_name,
                    top1_conf=top1_conf,
                    top3_list=top3_list,
                    latency_dict=latency_dict,
                    current_fps=fps_measured
                )
            else:
                rendered_frame = frame

            # Tính toán FPS tức thời
            t_loop_end = time.perf_counter()
            elapsed_sec = t_loop_end - t_loop_start

            # Khóa tốc độ 30 FPS nếu bật fps_lock_30
            if visualizer.fps_lock_30:
                target_interval = 1.0 / 30.0
                if elapsed_sec < target_interval:
                    time.sleep(target_interval - elapsed_sec)

            now = time.time()
            if now - t_fps_start >= 0.5:
                fps_measured = 1.0 / max(0.001, time.perf_counter() - t_loop_start)
                t_fps_start = now

            # Hiển thị cửa sổ đồ họa
            if display_gui:
                cv2.imshow(window_name, rendered_frame)
                key = cv2.waitKey(1) & 0xFF

                if key == ord('q'):
                    print("[Application] Nhận lệnh thoát từ người dùng (Key 'q').")
                    break
                elif key == ord('s'):
                    visualizer.save_snapshot(rendered_frame, top1_name, top1_conf)
                elif key == ord('d'):
                    visualizer.debug_mode = not visualizer.debug_mode
                    print(f"[Application] Chế độ Debug HUD: {'BẬT' if visualizer.debug_mode else 'TẮT'}")
                elif key == ord('f'):
                    visualizer.fps_lock_30 = not visualizer.fps_lock_30
                    print(f"[Application] Khóa tốc độ 30 FPS: {'BẬT' if visualizer.fps_lock_30 else 'TẮT'}")
                elif key == ord('p'):
                    visualizer.is_paused = not visualizer.is_paused
                    print(f"[Application] Trạng thái video: {'TẠM DỪNG' if visualizer.is_paused else 'TIẾP TỤC'}")

            # Kiểm tra giới hạn số khung hình kiểm thử
            if max_frames and frame_idx >= max_frames:
                print(f"[Application] Đã hoàn thành {max_frames} khung hình kiểm thử.")
                break

    finally:
        stream.stop()
        if display_gui:
            cv2.destroyAllWindows()
        print("[Application] Đã đóng ứng dụng và giải phóng tài nguyên phần cứng an toàn.")


# PHẦN 8: Thực Thi Ứng Dụng (Main Execution Block)


In [8]:
if __name__ == '__main__':
    # Chạy ứng dụng trực tiếp với camera USB
    run_live_video_classification(camera_index=0, max_frames=None, display_gui=True)



  KHỞI ĐỘNG ỨNG DỤNG REAL-TIME VIDEO CLASSIFICATION ViT (AMD KRIA KV260)  
[KV260 Engine] Quantization Backend: 'onednn' (Hỗ trợ: ['onednn'])


C:\Users\tuan2\AppData\Local\Temp\ipykernel_9740\2713266673.py:26: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantization.prepare_qat(qat_model, inplace=True)
C:\Users\tuan2\AppData\Local\Temp\ipykernel_9740\2713266673.py:27: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (tor

[KV260 Loader] ✅ Đã nạp thành công 'models_cache/vit_qat_int8.pth' (424 tensors).
[Camera] Tìm kiếm USB Webcam tại index 0...
[Camera] ✅ Kết nối thành công USB Webcam (1280x720 @ 30.0 FPS)

[Application] 🚀 Ứng dụng đã sẵn sàng! Đang phát luồng video...
              Bấm phím 'q' trên cửa sổ video để thoát.

[Application] Khóa tốc độ 30 FPS: BẬT
[Application] Khóa tốc độ 30 FPS: TẮT
[Application] Nhận lệnh thoát từ người dùng (Key 'q').
[Application] Đã đóng ứng dụng và giải phóng tài nguyên phần cứng an toàn.
